In [3]:
!pip install langchain langchain-community langchain-text-splitters
!pip install sentence-transformers faiss-cpu
!pip install pypdf
!pip install ollama
!pip install streamlit pandas plotly

In [11]:
import langchain
import langchain_community
import langchain_text_splitters
import sentence_transformers
import pypdf
import ollama
import faiss
import streamlit
import pandas
import plotly

In [14]:
knowledge_text = """
Python Programming Basics
Python is a high-level, general-purpose programming language known for its simple and readable syntax.

Variables:
A variable is a name used to store a value.
Example:
name = "Divya"
age = 26

Python Data Types:
Common Python data types include int, float, string, boolean, list, tuple, set, and dictionary.

Conditional Statements:
Python uses if, elif, and else statements to make decisions based on conditions.

Loops:
Python provides for loops and while loops for repeating a block of code.

Functions:
A function is a reusable block of code designed to perform a specific task.
Functions are created using the def keyword.

Lists:
A list is an ordered and changeable collection of items.
Example:
fruits = ["apple", "banana", "orange"]

Dictionaries:
A dictionary stores data as key-value pairs.
Example:
student = {"name": "Divya", "age": 26}

Exception Handling:
Python uses try and except blocks to handle errors without stopping the entire program.

Object-Oriented Programming:
Python supports object-oriented programming using classes and objects.

Machine Learning:
Machine learning is a branch of artificial intelligence in which computers learn patterns from data and use those patterns to make predictions or decisions.
"""

with open("documents/python_notes.txt", "w", encoding="utf-8") as file:
    file.write(knowledge_text)



In [15]:
import os
print("Files in documents folder:")
for file in os.listdir("documents"):
    print(file)

Files in documents folder:
python_notes.txt


In [16]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("documents/python_notes.txt", encoding="utf-8")
documents = loader.load()
print("Number of documents:", len(documents))
print("\nDocument content:\n")
print(documents[0].page_content)

Number of documents: 1

Document content:


Python Programming Basics
Python is a high-level, general-purpose programming language known for its simple and readable syntax.

Variables:
A variable is a name used to store a value.
Example:
name = "Divya"
age = 26

Python Data Types:
Common Python data types include int, float, string, boolean, list, tuple, set, and dictionary.

Conditional Statements:
Python uses if, elif, and else statements to make decisions based on conditions.

Loops:
Python provides for loops and while loops for repeating a block of code.

Functions:
A function is a reusable block of code designed to perform a specific task.
Functions are created using the def keyword.

Lists:
A list is an ordered and changeable collection of items.
Example:
fruits = ["apple", "banana", "orange"]

Dictionaries:
A dictionary stores data as key-value pairs.
Example:
student = {"name": "Divya", "age": 26}

Exception Handling:
Python uses try and except blocks to handle errors without s

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)
print("Number of chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content)

Number of chunks: 6

--- Chunk 1 ---
Python Programming Basics
Python is a high-level, general-purpose programming language known for its simple and readable syntax.

Variables:
A variable is a name used to store a value.
Example:
name = "Divya"
age = 26

--- Chunk 2 ---
Python Data Types:
Common Python data types include int, float, string, boolean, list, tuple, set, and dictionary.

Conditional Statements:
Python uses if, elif, and else statements to make decisions based on conditions.

--- Chunk 3 ---
Loops:
Python provides for loops and while loops for repeating a block of code.

Functions:
A function is a reusable block of code designed to perform a specific task.
Functions are created using the def keyword.

--- Chunk 4 ---
Lists:
A list is an ordered and changeable collection of items.
Example:
fruits = ["apple", "banana", "orange"]

Dictionaries:
A dictionary stores data as key-value pairs.
Example:
student = {"name": "Divya", "age": 26}

--- Chunk 5 ---
Exception Handling:
Pyt

In [19]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
texts = [chunk.page_content for chunk in chunks]
embeddings = embedding_model.embed_documents(texts)
print("Number of chunks:", len(texts))
print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))

Number of chunks: 6
Number of embeddings: 6
Embedding dimension: 384


In [22]:
from langchain_community.vectorstores import FAISS
vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)

In [23]:
vector_db.save_local("vector_db")

In [24]:
question = "What are Python data types?"
results = vector_db.similarity_search(question, k=2)
print("Question:", question)
for i, result in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(result.page_content)
    print()

Question: What are Python data types?
--- Result 1 ---
Python Data Types:
Common Python data types include int, float, string, boolean, list, tuple, set, and dictionary.

Conditional Statements:
Python uses if, elif, and else statements to make decisions based on conditions.

--- Result 2 ---
Python Programming Basics
Python is a high-level, general-purpose programming language known for its simple and readable syntax.

Variables:
A variable is a name used to store a value.
Example:
name = "Divya"
age = 26



In [25]:
import ollama
question = "What are Python data types?"
results = vector_db.similarity_search(question, k=2)
context = "\n\n".join([result.page_content for result in results])
print(context)

Python Data Types:
Common Python data types include int, float, string, boolean, list, tuple, set, and dictionary.

Conditional Statements:
Python uses if, elif, and else statements to make decisions based on conditions.

Python Programming Basics
Python is a high-level, general-purpose programming language known for its simple and readable syntax.

Variables:
A variable is a name used to store a value.
Example:
name = "Divya"
age = 26


In [26]:
prompt = f"""
You are a helpful question-answering assistant.
Answer the user's question using ONLY the information provided in the context below.

If the answer is not present in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response["message"]["content"]
print("Question:", question)
print(answer)

Question: What are Python data types?
Python data types include int, float, string, boolean, list, tuple, set, and dictionary.


In [27]:
question = "Who is the President of India?"
results = vector_db.similarity_search(question, k=2)
context = "\n\n".join([result.page_content for result in results])
prompt = f"""
You are a helpful question-answering assistant.

Answer the user's question using ONLY the information provided
in the context below.

If the answer is not present in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
answer = response["message"]["content"]
print("Question:", question)
print(answer)

Question: Who is the President of India?
I don't know based on the provided documents.


In [31]:
test_questions = [
    "What are Python data types?",
    "What is a Python function?",
    "What is a Python list?",
    "Who is the President of India?"
]
for question in test_questions:

    results = vector_db.similarity_search(
        question,
        k=2
    )

    context = "\n\n".join(
        result.page_content
        for result in results
    )

    
    print("QUESTION:", question)
    print(context)
    print()

QUESTION: What are Python data types?
Python Data Types:
Common Python data types include int, float, string, boolean, list, tuple, set, and dictionary.

Conditional Statements:
Python uses if, elif, and else statements to make decisions based on conditions.

Python Programming Basics
Python is a high-level, general-purpose programming language known for its simple and readable syntax.

Variables:
A variable is a name used to store a value.
Example:
name = "Divya"
age = 26

QUESTION: What is a Python function?
Loops:
Python provides for loops and while loops for repeating a block of code.

Functions:
A function is a reusable block of code designed to perform a specific task.
Functions are created using the def keyword.

Python Programming Basics
Python is a high-level, general-purpose programming language known for its simple and readable syntax.

Variables:
A variable is a name used to store a value.
Example:
name = "Divya"
age = 26

QUESTION: What is a Python list?
Python Data Types:

In [32]:
questions = [
    "What are Python data types?",
    "Who is the President of India?"
]
for question in questions:

    results = vector_db.similarity_search(
        question,
        k=2
    )

    context = "\n\n".join(
        result.page_content
        for result in results
    )

    prompt = f"""
Answer the question using ONLY the context provided.

If the answer is not in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response["message"]["content"]

   
    print("QUESTION:", question)
    print("ANSWER:", answer)

QUESTION: What are Python data types?
ANSWER: Python data types include int, float, string, boolean, list, tuple, set, and dictionary.
QUESTION: Who is the President of India?
ANSWER: I don't know based on the provided documents.


In [34]:
import pandas as pd

log_file = "logs/chat_logs.csv"

df = pd.read_csv(log_file)

print("Available columns:")
print(df.columns.tolist())

average_retrieval_time = df["retrieval_time"].mean()

average_total_latency = df["total_latency"].mean()

print("Number of Queries:", len(df))

print(
    "Average Retrieval Time:",
    round(average_retrieval_time, 4),
    "seconds"
)

print(
    "Average Total Latency:",
    round(average_total_latency, 4),
    "seconds"
)

if "generation_time_seconds" in df.columns:

    average_generation_time = df[
        "generation_time_seconds"
    ].mean()

    print(
        "Average Generation Time:",
        round(average_generation_time, 4),
        "seconds"
    )

if "total_tokens" in df.columns:

    total_tokens = df[
        "total_tokens"
    ].sum()

    average_tokens = df[
        "total_tokens"
    ].mean()

    print(
        "Total Tokens Used:",
        int(total_tokens)
    )

    print(
        "Average Tokens Per Query:",
        round(average_tokens, 2)
    )

Available columns:
['timestamp', 'question', 'retrieved_context', 'answer', 'retrieval_time', 'total_latency', 'feedback']
Number of Queries: 1
Average Retrieval Time: 0.0563 seconds
Average Total Latency: 22.1874 seconds


In [35]:
import pandas as pd
df = pd.read_csv("logs/chat_logs.csv")
positive = (df["feedback"] == "positive").sum()
negative = (df["feedback"] == "negative").sum()
not_provided = (df["feedback"] == "not_provided").sum()
print("Positive Feedback:", positive)
print("Negative Feedback:", negative)
print("Feedback Not Provided:", not_provided)

Positive Feedback: 0
Negative Feedback: 0
Feedback Not Provided: 1
